# Connect Four: MCTS vs Minimax
### Strategic Search — From Monte Carlo Tree Search to Alpha-Beta Pruning

---

**Architecture** — four clean, decoupled modules:

| Module | Responsibility |
|---|---|
| `connect_four.py` | Game engine: board, gravity, win detection |
| `mcts_agent.py` | MCTS: 4-phase loop + UCT formula |
| `minimax_agent.py` | Minimax + Alpha-Beta pruning |
| `visualization.py` | Board renderer, tree visualizer, charts |

---

In [ ]:
import math, random, time, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# Make sure the module files are on the path
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from connect_four  import ConnectFourBoard, P1, P2, ROWS, COLS
from mcts_agent    import MCTSAgent, MCTSNode
from minimax_agent import MinimaxAgent
from visualization import (draw_board, animate_game, draw_mcts_tree,
                            draw_uct_analysis, draw_tree_growth, draw_comparison)

# Dark aesthetic for all matplotlib plots
BG, PAN, GOLD = '#0d0d14', '#13131e', '#f7c948'
plt.rcParams.update({
    'figure.facecolor': BG,  'axes.facecolor': PAN,
    'axes.edgecolor':  '#2a2a40', 'text.color':      '#e8e0c8',
    'axes.labelcolor': '#e8e0c8', 'xtick.color':     '#6a6a8a',
    'ytick.color':     '#6a6a8a', 'grid.color':      '#1e1e2e',
    'grid.linestyle':  '--',      'font.family':     'monospace',
})
random.seed(42); np.random.seed(42)
print('All modules loaded.')

---
## 1  The Game Engine

Connect Four is a **perfect information, zero-sum, two-player** game:
- 6 rows × 7 columns
- **Gravity** — pieces fall to the lowest empty row in a column
- **Win** — first to place 4 in a row (horizontal, vertical, or diagonal)
- ~**10¹³ possible states** — exhaustive search is impossible

The engine is completely decoupled from AI logic. It knows only the rules.

In [ ]:
# ── Quick sanity check of the engine ──────────────────────────────────
b = ConnectFourBoard()

# Simulate a few moves
moves = [3, 3, 4, 2, 3, 3]   # P1 builds a column-3 stack
for col in moves:
    b.drop(col)

print('Board after 6 moves:')
print(b)
print(f'Legal moves  : {b.legal_moves()}')
print(f'Current turn : Player {b.current}')
print(f'Terminal?    : {b.is_terminal()}')
print(f'Winner?      : {b.winner}')

# Draw the board
draw_board(b, title='After 6 Moves — Engine Sanity Check')

In [ ]:
# ── Test win detection in all 4 directions ─────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('Win Detection — All 4 Directions', color=GOLD, fontsize=13)

scenarios = [
    ('Horizontal', [3, 0, 4, 0, 5, 0, 6]),
    ('Vertical',   [2, 3, 2, 3, 2, 3, 2]),
    ('Diagonal ↗', [0, 1, 1, 2, 2, 3, 2, 3, 3, 6, 3]),
    ('Diagonal ↘', [6, 5, 5, 4, 4, 3, 4, 3, 3, 0, 3]),
]

for ax, (name, moves_seq) in zip(axes, scenarios):
    board = ConnectFourBoard()
    for col in moves_seq:
        if not board.is_terminal():
            board.drop(col)
    draw_board(board, title=name, ax=ax)

plt.tight_layout(); plt.show()
print('All 4 win directions verified.')

---
## 2  The MCTS Algorithm

Monte Carlo Tree Search replaces human-written heuristics with **statistical sampling**.
It repeats a 4-phase loop thousands of times, building a probability estimate of which moves lead to victory.

| Phase | What happens | Connect Four analogy |
|---|---|---|
| **① Selection** | Walk tree via UCT until expandable leaf | Follow the most-promising column sequence already tested |
| **② Expansion** | Add one new child for an untried column | Try one unexplored move from that position |
| **③ Simulation** | Random rollout → Win / Loss / Draw | Fast-forward a random game to the end |
| **④ Backprop** | Update N and Q all the way to root | Tell every ancestor how this random game went |

### UCT Formula

$$\text{UCT}(v) = \underbrace{\frac{Q(v)}{N(v)}}_{\text{exploit: win rate}} + C \cdot \underbrace{\sqrt{\frac{\ln N(\text{parent})}{N(v)}}}_{\text{explore: visit rarely}}$$

- **Q/N** — average win rate of simulations through this node
- **C·√(...)** — exploration bonus that *shrinks* as the node is visited more  
- **C = √2** is theoretically optimal for rewards in [0, 1]
- **Final decision**: pick the child with the **most visits** (N), not highest UCT — more statistically robust

In [ ]:
# ── Single MCTS search from start position ─────────────────────────────
board = ConnectFourBoard()
agent = MCTSAgent(n_iterations=600)

t0 = time.time()
move, root = agent.search(board)
elapsed = time.time() - t0

print(f'Search (600 iterations): best move = col {move}  [{elapsed*1000:.0f}ms]')
print(f'Total nodes in tree    : {sum(1 for _ in root.children)}')
print()
print(f'  {"Column":>8}  {"N (visits)":>10}  {"Q/N (win%)":>12}  {"UCT":>8}')
print('  ' + '-'*46)
for c in sorted(root.children, key=lambda x: x.N, reverse=True):
    print(f'  {f"col {c.move}":>8}  {c.N:>10}  {c.value*100:>11.1f}%  {c.uct():>8.3f}')

print(f'\n  → MCTS chooses col {move} (most visited = most reliable)')

In [ ]:
# ── UCT Breakdown: what makes MCTS choose what it chooses ──────────────
draw_uct_analysis(root)

In [ ]:
# ── MCTS Tree after 600 iterations ────────────────────────────────────
# Node size ∝ visits  |  Color = win rate  |  Gold path = best line
draw_mcts_tree(root, max_depth=3,
               title='MCTS Search Tree — Start Position  (600 iterations)')

In [ ]:
# ── How the tree grows with more iterations ────────────────────────────
# As N→∞, MCTS converges to the same answer as perfect Minimax.
# With 5 iters: a rough sketch. With 500: a near-optimal strategy.
board_fresh = ConnectFourBoard()
draw_tree_growth(board_fresh, iter_counts=(5, 25, 100, 500))

---
## 3  Exploration vs Exploitation — The UCT Tradeoff

The constant **C** controls how much the agent explores unknown moves vs exploits known good ones.
- **High C** → visits every branch almost equally (curious but slow to converge)
- **Low C** → always returns to the best-known move (greedy, may miss surprises)
- **C = √2** — the theoretical sweet spot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('UCT: Exploration vs Exploitation', fontsize=13, color=GOLD)

# Left: UCT curve for different C values
ax = axes[0]; ax.set_facecolor(PAN)
Nr = np.arange(1, 101); Np = 100; Qm = 0.55
for Cv, col, lab in [
    (0.5,            '#ff6b6b', 'C=0.5  (exploit-heavy)'),
    (1.0,            '#74b9ff', 'C=1.0'),
    (math.sqrt(2),   '#f7c948', 'C=√2   (theoretical optimum)'),
    (2.0,            '#38d9a9', 'C=2.0  (explore-heavy)'),
]:
    ucb = Qm + Cv * np.sqrt(np.log(Np) / Nr)
    ax.plot(Nr, ucb, color=col, lw=2.2, label=lab)
ax.axhline(Qm, color='white', ls=':', lw=1, alpha=0.4, label='exploit only')
ax.set_xlabel('Node visits (N)'); ax.set_ylabel('UCT score')
ax.set_title('UCT Score vs Visit Count', color=GOLD)
ax.legend(fontsize=8, facecolor=PAN, edgecolor='#2a2a40', labelcolor='#e8e0c8')
ax.grid(alpha=0.25)

# Right: 3 nodes competing under UCT (simulate bandit problem)
ax2 = axes[1]; ax2.set_facecolor(PAN)
nodes_sim = [
    {'Q': 0, 'N': 0, 'true': 0.75, 'color': GOLD,    'label': 'Column A (true win=75%)'},
    {'Q': 0, 'N': 0, 'true': 0.50, 'color': '#38d9a9','label': 'Column B (true win=50%)'},
    {'Q': 0, 'N': 0, 'true': 0.25, 'color': '#ff6b6b','label': 'Column C (true win=25%)'},
]
parent_N = 0; visit_h = [[] for _ in nodes_sim]
for _ in range(400):
    parent_N += 1
    scores = []
    for n in nodes_sim:
        if n['N'] == 0: scores.append(float('inf'))
        else:           scores.append(n['Q']/n['N'] + math.sqrt(2)*math.sqrt(math.log(parent_N)/n['N']))
    chosen = scores.index(max(scores))
    r = np.random.normal(nodes_sim[chosen]['true'], 0.08)
    nodes_sim[chosen]['N'] += 1; nodes_sim[chosen]['Q'] += r
    for i, n in enumerate(nodes_sim): visit_h[i].append(n['N'])

for i, (n, h) in enumerate(zip(nodes_sim, visit_h)):
    ax2.plot(h, color=n['color'], lw=2, label=n['label'])
ax2.set_xlabel('Iterations'); ax2.set_ylabel('Cumulative visits')
ax2.set_title('UCT Naturally Favours Better Columns', color=GOLD)
ax2.legend(fontsize=8, facecolor=PAN, edgecolor='#2a2a40', labelcolor='#e8e0c8')
ax2.grid(alpha=0.25)

plt.tight_layout(); plt.show()
print(f"Final: A={nodes_sim[0]['N']}  B={nodes_sim[1]['N']}  C={nodes_sim[2]['N']}")
print('UCT correctly allocated most visits to the best column.')

---
## 4  MCTS Agent vs Human — Live Game

Watch a full game unfold with the MCTS agent making real decisions at each step.
The board updates after every move and the MCTS tree is shown at the first decision.

In [ ]:
def run_mcts_vs_random(n_iter=500, seed=42, verbose=True):
    """
    MCTS (P1, Red) vs Random (P2, Yellow).
    Returns history of board states.
    """
    random.seed(seed)
    board   = ConnectFourBoard()
    agent   = MCTSAgent(n_iterations=n_iter)
    history = [board.clone()]
    first_root = None

    while not board.is_terminal():
        if board.current == P1:
            move, root = agent.search(board)
            if first_root is None:
                first_root = root
            who = 'MCTS (Red)'
        else:
            move = random.choice(board.legal_moves())
            who  = 'Random (Yellow)'

        board.drop(move)
        history.append(board.clone())

        if verbose:
            status = ''
            if board.winner:   status = f'  ← WIN'
            elif board.is_draw: status = '  ← DRAW'
            print(f'  Move {board.move_count:>2}: {who:18s} → col {move}{status}')

    outcome = ('Red wins!' if board.winner == P1 else
               'Yellow wins!' if board.winner == P2 else 'Draw')
    if verbose:
        print(f'\n  Result: {outcome}  ({board.move_count} moves)')

    return history, first_root, board


history, first_root, final_board = run_mcts_vs_random(n_iter=500)

In [ ]:
# Draw first move decision + final board side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor(BG)
draw_board(history[1], title='MCTS First Move Decision', ax=axes[0])
draw_board(final_board, title='Final Board State', ax=axes[1])
plt.tight_layout(); plt.show()

In [ ]:
# MCTS decision tree at move 1
draw_mcts_tree(first_root, max_depth=3,
               title='MCTS Decision Tree — Move 1  (500 iterations)')

In [ ]:
# Animate the full game (renders inline in Jupyter)
from IPython.display import HTML
anim = animate_game(history, interval=800, title='MCTS (Red) vs Random (Yellow)')
HTML(anim.to_jshtml())

---
## 5  MCTS vs Baselines — Statistical Analysis

Run N games for each agent against a random opponent and compare win rates.

In [ ]:
def run_agent_vs_random(agent_fn, n_games=30, player=P1, seed_start=0):
    """
    Run `n_games` games of `agent_fn` (callable: board → col) as `player`
    against a random opponent. Returns (wins, losses, draws).
    """
    wins = losses = draws = 0
    for seed in range(seed_start, seed_start + n_games):
        random.seed(seed)
        board = ConnectFourBoard()
        while not board.is_terminal():
            if board.current == player:
                col = agent_fn(board)
            else:
                col = random.choice(board.legal_moves())
            board.drop(col)
        if board.winner == player:       wins   += 1
        elif board.winner is not None:   losses += 1
        else:                            draws  += 1
    return wins, losses, draws


N_GAMES = 30
print(f'Running {N_GAMES} games each vs Random opponent...')
print('(This takes ~60s — MCTS runs 400 iterations per move)\n')

# Random baseline
random_fn  = lambda b: random.choice(b.legal_moves())
# Greedy baseline: always play center-most available column
from connect_four import COLS
CENTER_PREF = [3, 2, 4, 1, 5, 0, 6]
greedy_fn  = lambda b: next(c for c in CENTER_PREF if c in b.legal_moves())
# MCTS agent
mcts_agent = MCTSAgent(n_iterations=400)
mcts_fn    = lambda b: mcts_agent.search(b)[0]

results_table = {}
for name, fn in [('Random', random_fn), ('Greedy', greedy_fn), ('MCTS', mcts_fn)]:
    w, l, d = run_agent_vs_random(fn, N_GAMES)
    results_table[name] = {'wins': w, 'losses': l, 'draws': d}
    print(f'  {name:8s}  wins={w:3d} ({w/N_GAMES*100:.0f}%)  '
          f'losses={l:3d}  draws={d:3d}')

In [ ]:
draw_comparison(results_table)

---
## 6  BONUS — Minimax + Alpha-Beta vs MCTS Live Match

### How Minimax works

Minimax assumes both players play **perfectly**:
- **MAX** (the agent) picks the move with the **highest** score
- **MIN** (the opponent) picks the move with the **lowest** score for you

$$V(s) = \max\bigl(\min\bigl(V(s')\bigr)\bigr)$$

**Alpha-Beta Pruning** skips branches that cannot change the result:
- α = best score MAX already has → prune if child score ≤ α (MIN won't pick it)
- β = best score MIN already has → prune if child score ≥ β (MAX won't pick it)
- Prune when **α ≥ β** → reduces O(b^d) to O(b^(d/2)) in the best case

In [ ]:
def run_mcts_vs_minimax(mcts_iters=600, mm_depth=5, seed=7, verbose=True):
    """
    MCTS (P1, Red) vs Minimax+AlphaBeta (P2, Yellow).
    Prints every move and shows the board at key moments.
    """
    random.seed(seed)
    board     = ConnectFourBoard()
    mcts      = MCTSAgent(n_iterations=mcts_iters)
    minimax   = MinimaxAgent(depth=mm_depth, player=P2)
    history   = [board.clone()]
    roots     = []   # store MCTS roots for visualization

    if verbose:
        print('═'*55)
        print(f'  MCTS ({mcts_iters} iters)  vs  Minimax (depth {mm_depth})')
        print(f'  Red = MCTS  |  Yellow = Minimax')
        print('═'*55)

    while not board.is_terminal():
        t0 = time.time()
        if board.current == P1:
            move, root = mcts.search(board)
            roots.append(root)
            who   = 'MCTS    (Red)'
        else:
            move = minimax.search(board)
            who  = f'Minimax (Yellow)  [{minimax.nodes_searched:,} nodes]'

        board.drop(move)
        history.append(board.clone())
        elapsed = (time.time() - t0) * 1000

        if verbose:
            status = ''
            if board.winner:    status = '  ← WIN'
            elif board.is_draw: status = '  ← DRAW'
            print(f'  [{board.move_count:>2}] {who:35s} col={move}  {elapsed:.0f}ms{status}')

    outcome = ('Red (MCTS) wins!'      if board.winner == P1 else
               'Yellow (Minimax) wins!' if board.winner == P2 else 'Draw!')
    if verbose:
        print('═'*55)
        print(f'  RESULT: {outcome}')
        print('═'*55)

    return history, roots, board, outcome


history_match, roots_match, final_match, outcome_match = run_mcts_vs_minimax(
    mcts_iters=600, mm_depth=5
)

In [ ]:
# Show the final board and the MCTS tree from move 1
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor(BG)
draw_board(history_match[1], title='After Move 1 — MCTS Decision', ax=axes[0])
draw_board(final_match, title=f'Final Board — {outcome_match}', ax=axes[1])
plt.tight_layout(); plt.show()

In [ ]:
# MCTS tree at the first move of the match
if roots_match:
    draw_mcts_tree(roots_match[0], max_depth=3,
                   title='MCTS Decision Tree — Match Move 1')

In [ ]:
# Animate the full MCTS vs Minimax match
from IPython.display import HTML
anim_match = animate_game(
    history_match, interval=900,
    title='MCTS (Red) vs Minimax+αβ (Yellow)'
)
HTML(anim_match.to_jshtml())

In [ ]:
# ── Tournament: MCTS vs Minimax, multiple seeds ────────────────────────
N_MATCH = 10   # 10 games (Minimax is slower)
print(f'Tournament: {N_MATCH} games  MCTS vs Minimax')

mcts_wins = mm_wins = draws = 0
for seed in range(N_MATCH):
    random.seed(seed)
    board   = ConnectFourBoard()
    mcts    = MCTSAgent(n_iterations=500)
    minimax = MinimaxAgent(depth=4, player=P2)
    while not board.is_terminal():
        if board.current == P1:
            col, _ = mcts.search(board)
        else:
            col = minimax.search(board)
        board.drop(col)
    if board.winner == P1:        mcts_wins += 1
    elif board.winner == P2:      mm_wins   += 1
    else:                         draws     += 1
    print(f'  Game {seed+1}: {"MCTS" if board.winner==P1 else "Minimax" if board.winner==P2 else "Draw"}')

print(f'\n  MCTS wins  : {mcts_wins}/{N_MATCH}')
print(f'  Minimax wins: {mm_wins}/{N_MATCH}')
print(f'  Draws       : {draws}/{N_MATCH}')

draw_comparison({
    'MCTS':    {'wins': mcts_wins, 'losses': mm_wins,   'draws': draws},
    'Minimax': {'wins': mm_wins,   'losses': mcts_wins, 'draws': draws},
})

---
## 7  Key Takeaways

| | Minimax + α-β | MCTS |
|---|---|---|
| **Requires heuristic** | Yes — must score mid-game positions | No — learns from game outcomes |
| **Complexity** | O(b^d) → O(b^(d/2)) with pruning | O(N) iterations, any budget |
| **Optimal** | Yes, if depth is enough | Converges to optimal as N→∞ |
| **Works for Go** | No (heuristic impossible) | Yes (AlphaGo used MCTS) |
| **Anytime** | No (fixed depth commitment) | Yes (stop anytime for a best guess) |

The deeper insight: **both algorithms solve the same problem differently.**  
Minimax exhausts the tree logically. MCTS approximates it statistically.  
As MCTS iterations N→∞, they converge to the identical answer.